In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Read the supplied dataset
DATA_FILE = "spread_locator_dataset.xlsx"
df = pd.read_excel(DATA_FILE)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Basic inspection
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

print("\nTransaction status:")
print(df["transaction_status"].value_counts())

print("\nTransaction amount summary:")
display(df["transaction_amount"].describe())


# 1. Bernoulli Distribution – Transaction Occurrence

For this analysis, **transaction success** is treated as the Bernoulli outcome:

- Success = 1
- Fail = 0

This is a practical interpretation of transaction occurrence because the supplied dataset explicitly records `transaction_status` as Success/Fail.

The estimated probability of success is the observed success proportion.


In [ ]:
# Bernoulli fit using transaction status
df["success"] = (df["transaction_status"] == "Success").astype(int)

p_success = df["success"].mean()

print(f"Estimated Bernoulli p (Success): {p_success:.4f}")
print(f"Observed success rate: {p_success*100:.2f}%")

# Compare observed and theoretical probabilities
bernoulli_table = pd.DataFrame({
    "Outcome": ["Fail (0)", "Success (1)"],
    "Observed Probability": [1-p_success, p_success]
})
display(bernoulli_table)


### Interpretation

The estimated Bernoulli success probability is about **44.55%**. In this sample, a randomly selected transaction record has roughly a 44.55% chance of being marked as successful.


# 2. Binomial Distribution – Weekly Transaction Count

The supplied `transaction_count` field represents the number of transactions made by a customer in a given week.

Because a Binomial distribution requires a fixed number of trials, this practical uses the observed maximum count (**n = 9**) as the fixed trial count and estimates \(p\) from the sample mean:

\[
p = 
rac{ar{x}}{n}
\]

This is an analytical approximation for the supplied data; the assignment does not specify a separate number of weekly trials.


In [ ]:
# Binomial fit to transaction_count
counts = df["transaction_count"].astype(int)

n_binom = int(counts.max())
p_binom = counts.mean() / n_binom
p_binom = min(max(p_binom, 0), 1)

print(f"Binomial n: {n_binom}")
print(f"Estimated p: {p_binom:.4f}")
print(f"Observed mean weekly count: {counts.mean():.4f}")
print(f"Binomial expected count (n*p): {n_binom*p_binom:.4f}")

# Observed vs fitted PMF
x = np.arange(0, n_binom + 1)
observed_pmf = counts.value_counts(normalize=True).reindex(x, fill_value=0)
fitted_pmf = stats.binom.pmf(x, n_binom, p_binom)

binom_compare = pd.DataFrame({
    "Count": x,
    "Observed Probability": observed_pmf.values,
    "Fitted Binomial PMF": fitted_pmf
})
display(binom_compare)


In [ ]:
plt.figure(figsize=(8,5))
width = 0.38
plt.bar(x - width/2, observed_pmf.values, width=width, label="Observed")
plt.bar(x + width/2, fitted_pmf, width=width, label="Binomial fit")
plt.xlabel("Weekly Transaction Count")
plt.ylabel("Probability")
plt.title("Observed vs Binomial Distribution")
plt.legend()
plt.tight_layout()
plt.show()


# 3. Poisson Distribution – Transactions per Day

For Poisson analysis, the number of transaction records is aggregated by `transaction_date`. The mean number of records per day is used as the Poisson rate parameter \(\lambda\).

\[
\lambda = 	ext{mean daily transaction count}
\]


In [ ]:
daily_transactions = df.groupby("transaction_date").size()
lambda_poisson = daily_transactions.mean()

print(f"Number of days: {len(daily_transactions)}")
print(f"Estimated Poisson lambda: {lambda_poisson:.4f}")

daily_x = np.arange(daily_transactions.min(), daily_transactions.max()+1)
poisson_pmf = stats.poisson.pmf(daily_x, lambda_poisson)

poisson_table = pd.DataFrame({
    "Daily Count": daily_x,
    "Observed Frequency": daily_transactions.value_counts().reindex(daily_x, fill_value=0).values,
    "Poisson PMF": poisson_pmf
})
display(poisson_table)

plt.figure(figsize=(8,5))
plt.bar(daily_x, poisson_pmf, alpha=0.7)
plt.xlabel("Transactions per Day")
plt.ylabel("Poisson Probability")
plt.title("Poisson Distribution Fit – Daily Transactions")
plt.tight_layout()
plt.show()


### Interpretation

The estimated daily Poisson rate is approximately **7.10 transactions per day**. The Poisson model provides a simple count-based description of the number of transaction records observed on each day.


# 4. Log-Normal and Power Law Models – Transaction Amounts

Transaction amounts are positive and strongly right-skewed in the supplied sample, making positive skewed distributions appropriate candidates.

We fit:

- Log-Normal distribution
- Power Law / Pareto distribution

The models are compared using the **Kolmogorov–Smirnov (K-S) statistic**, p-value, and AIC. Lower K-S and AIC indicate a better fit among the models considered.


In [ ]:
amounts = df["transaction_amount"].astype(float)

# Log-normal fit
ln_shape, ln_loc, ln_scale = stats.lognorm.fit(amounts, floc=0)
ln_loglik = np.sum(stats.lognorm.logpdf(amounts, ln_shape, loc=ln_loc, scale=ln_scale))
ln_aic = 2*2 - 2*ln_loglik
ln_ks = stats.kstest(amounts, "lognorm", args=(ln_shape, ln_loc, ln_scale))

# Pareto / power-law fit
pa_shape, pa_loc, pa_scale = stats.pareto.fit(amounts, floc=0)
pa_loglik = np.sum(stats.pareto.logpdf(amounts, pa_shape, loc=pa_loc, scale=pa_scale))
pa_aic = 2*2 - 2*pa_loglik
pa_ks = stats.kstest(amounts, "pareto", args=(pa_shape, pa_loc, pa_scale))

fit_table = pd.DataFrame({
    "Model": ["Log-Normal", "Power Law / Pareto"],
    "K-S Statistic": [ln_ks.statistic, pa_ks.statistic],
    "K-S p-value": [ln_ks.pvalue, pa_ks.pvalue],
    "AIC": [ln_aic, pa_aic]
})
display(fit_table)

print(f"Log-normal shape = {ln_shape:.4f}, scale = {ln_scale:.2f}")
print(f"Pareto shape = {pa_shape:.4f}, scale = {pa_scale:.2f}")


### Interpretation

For this dataset, the **Log-Normal model fits substantially better** than the fitted Pareto model. The log-normal K-S statistic is much smaller and its K-S p-value is high, while the Pareto fit has a much larger K-S statistic and an extremely small p-value.

Therefore, among these two candidates, the transaction amounts are better represented by a **Log-Normal distribution**.


# 5. Q-Q Plot – Test for Normality

A Q-Q plot is generated for the raw transaction amounts. Strong curvature or systematic deviation from the reference line indicates that the raw data do not closely follow a normal distribution.


In [ ]:
plt.figure(figsize=(7,6))
stats.probplot(amounts, dist="norm", plot=plt)
plt.title("Q-Q Plot of Transaction Amounts")
plt.tight_layout()
plt.show()

shapiro_stat, shapiro_p = stats.shapiro(amounts)
print(f"Shapiro-Wilk statistic: {shapiro_stat:.4f}")
print(f"Shapiro-Wilk p-value: {shapiro_p:.6g}")


### Interpretation

The raw transaction amounts are strongly right-skewed, so the Q-Q plot is expected to deviate from a straight line. The normality test also provides statistical evidence against normality for this sample.

This supports considering a transformation or a positively skewed distribution such as the log-normal model.


# 6. Box-Cox Transformation

The Box-Cox transformation is applied to the positive transaction amounts. The transformation chooses a value of \(\lambda\) that improves the shape of the data for statistical analysis.


In [ ]:
boxcox_values, boxcox_lambda = stats.boxcox(amounts)

print(f"Estimated Box-Cox lambda: {boxcox_lambda:.4f}")
print(f"Original skewness: {amounts.skew():.4f}")
print(f"Transformed skewness: {pd.Series(boxcox_values).skew():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12,5))

axes[0].hist(amounts, bins=25)
axes[0].set_title("Original Transaction Amounts")
axes[0].set_xlabel("Transaction Amount")
axes[0].set_ylabel("Frequency")

axes[1].hist(boxcox_values, bins=25)
axes[1].set_title("After Box-Cox Transformation")
axes[1].set_xlabel("Transformed Value")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


### Interpretation

The estimated Box-Cox parameter is approximately **−0.181**. The transformation reduces the strong positive skew in the original transaction amounts, producing a distribution that is much more symmetric.


# 7. Z-score and Probability of a Transaction Exceeding ₹5,000

The z-score for ₹5,000 is calculated using the sample mean and sample standard deviation:

\[
z = 
rac{x-ar{x}}{s}
\]

The normal-distribution probability \(P(X>5000)\) is then calculated from that z-score.

Because the raw transaction amounts are skewed, this normal-based probability should be treated as an approximation. A fitted log-normal probability and the empirical sample proportion are also reported for comparison.


In [ ]:
mean_amount = amounts.mean()
std_amount = amounts.std(ddof=1)
threshold = 5000

z_5000 = (threshold - mean_amount) / std_amount
normal_probability = stats.norm.sf(z_5000)
empirical_probability = (amounts > threshold).mean()
lognormal_probability = stats.lognorm.sf(
    threshold, ln_shape, loc=ln_loc, scale=ln_scale
)

print(f"Mean transaction amount: ₹{mean_amount:,.2f}")
print(f"Sample standard deviation: ₹{std_amount:,.2f}")
print(f"Z-score for ₹5,000: {z_5000:.4f}")
print(f"Normal-based P(X > ₹5,000): {normal_probability:.4%}")
print(f"Empirical P(X > ₹5,000): {empirical_probability:.4%}")
print(f"Log-normal fitted P(X > ₹5,000): {lognormal_probability:.4%}")
print(f"Observed transactions above ₹5,000: {(amounts > threshold).sum()} of {len(amounts)}")


### Interpretation

₹5,000 is about **0.823 standard deviations above the sample mean**. Under a normal approximation, the probability of exceeding ₹5,000 is about **20.52%**.

However, because the transaction amounts are right-skewed and are better described by a log-normal model, the fitted log-normal probability is about **13.84%**, while the observed sample proportion is **11.36% (25 of 220 records)**.

The difference illustrates why checking the distribution before applying a normal probability model is important.


# 8. PDF and CDF of Transaction Amounts

The fitted Log-Normal distribution is used to show:

- **PDF:** relative density around each transaction amount.
- **CDF:** probability that transaction amount is less than or equal to a given value.


In [ ]:
x_grid = np.linspace(amounts.min(), amounts.max(), 500)
pdf_values = stats.lognorm.pdf(x_grid, ln_shape, loc=ln_loc, scale=ln_scale)
cdf_values = stats.lognorm.cdf(x_grid, ln_shape, loc=ln_loc, scale=ln_scale)

fig, axes = plt.subplots(1, 2, figsize=(13,5))

axes[0].hist(amounts, bins=25, density=True, alpha=0.45, label="Observed")
axes[0].plot(x_grid, pdf_values, linewidth=2, label="Log-Normal PDF")
axes[0].set_title("Transaction Amount – PDF")
axes[0].set_xlabel("Transaction Amount (₹)")
axes[0].set_ylabel("Density")
axes[0].legend()

axes[1].plot(x_grid, cdf_values, linewidth=2)
axes[1].set_title("Transaction Amount – CDF")
axes[1].set_xlabel("Transaction Amount (₹)")
axes[1].set_ylabel("Cumulative Probability")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()


### Interpretation

The PDF shows where transaction amounts are most concentrated under the fitted log-normal model. The CDF increases from values near 0 toward 1 and can be used to estimate the probability that a transaction amount falls below a chosen threshold.


# 9. Final Distribution Judgement

### Best-fitting model for transaction amounts: **Log-Normal**

The log-normal distribution is preferred because:

1. Transaction amounts are strictly positive.
2. The raw data are strongly right-skewed.
3. The fitted log-normal model has a very small K-S statistic.
4. The log-normal K-S p-value is high for this sample.
5. The fitted Pareto model performs substantially worse under the same comparison.
6. The Box-Cox transformation also shows that the original amounts have a strong skew that can be reduced by transformation.

### Decision-making insight

For this dataset, a log-normal model is more suitable than a normal model for describing transaction amounts. This can help an e-commerce business estimate transaction probabilities while accounting for the long right tail of spending values.


In [ ]:
# Compact final results table
results = pd.DataFrame({
    "Metric": [
        "Number of records",
        "Mean transaction amount",
        "Transaction amount skewness",
        "Bernoulli success probability",
        "Poisson daily lambda",
        "Log-normal K-S statistic",
        "Log-normal K-S p-value",
        "Pareto K-S statistic",
        "Pareto K-S p-value",
        "Box-Cox lambda",
        "Z-score for ₹5,000",
        "Normal P(X > ₹5,000)",
        "Empirical P(X > ₹5,000)",
        "Log-normal P(X > ₹5,000)"
    ],
    "Value": [
        len(df),
        f"₹{mean_amount:,.2f}",
        f"{amounts.skew():.4f}",
        f"{p_success:.4f}",
        f"{lambda_poisson:.4f}",
        f"{ln_ks.statistic:.4f}",
        f"{ln_ks.pvalue:.6f}",
        f"{pa_ks.statistic:.4f}",
        f"{pa_ks.pvalue:.6g}",
        f"{boxcox_lambda:.4f}",
        f"{z_5000:.4f}",
        f"{normal_probability:.4%}",
        f"{empirical_probability:.4%}",
        f"{lognormal_probability:.4%}"
    ]
})
display(results)


# Conclusion

The analysis of the supplied Spread Locator dataset shows that different distribution models are useful for different types of transaction behavior.

- **Bernoulli** describes the binary transaction status.
- **Binomial** provides an approximate model for the weekly transaction-count field under a fixed-trial assumption.
- **Poisson** models the number of transaction records observed per day.
- **Log-Normal** is the strongest candidate among the tested models for transaction amounts.
- The raw transaction amounts are not well described by a normal distribution.
- **Box-Cox** substantially reduces skewness.
- A z-score can be used to calculate a normal-based probability, but distribution checking shows why a log-normal probability is more appropriate for this skewed dataset.
- PDF and CDF provide useful ways to understand transaction amount density and cumulative probability.

Overall, the project demonstrates how probability distributions and transformations can turn transaction data into practical statistical insights.
